In [7]:
%load_ext IPython.extensions.autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')


import sys
sys.path.append('../')
from model import FinData
from model import train_valid_test_split
from model import CatboostFinModel, SVMFinModel

import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
import pandas as pd
import numpy as np

import json
import datetime as dt
import pandas as pd
from sklearn.metrics import precision_score, recall_score, fbeta_score
import optuna 

The IPython.extensions.autoreload extension is already loaded. To reload it, use:
  %reload_ext IPython.extensions.autoreload


In [8]:
args1 = {
    "iterations": 10000,
    # "depth": 5,
    "learning_rate": 0.02,
    "use_best_model": True,
    # "l2_leaf_reg": 200,
    "loss_function": 'Logloss',
    "eval_metric": 'Logloss',
    "cat_features": [],
    "random_state": 42,
    # "class_weights": [1, 0.25], 
    "verbose": 0,
    "early_stopping_rounds": 500
}


In [9]:
company_name = 'Gazprom'
df_path = '../datasets/' + company_name + '_1_min.csv'

start_time = dt.datetime(2024, 2, 1)
end_time = dt.datetime(2024, 4, 30)
cutoff_time = start_time - dt.timedelta(days=90)

findata = FinData(df_path)
findata.restrict_time_down(cutoff_time)
findata.restrict_time_up(end_time + dt.timedelta(days=2))
findata.insert_all()

cat_feats = findata.cat_features
num_feats = findata.numeric_features
target = 'direction_binary_0'

data = findata.df

In [ ]:
def objective(trial):
    weight_1 = trial.suggest_float("class_weight_1", 0.05, 1.0)
    params={
        "iterations" : 10000,
        "learning_rate" : 0.02,
        "use_best_model" : True,
        "loss_function" : 'Logloss',
        "eval_metric" : 'Logloss',
        "cat_features" : [], 
        "random_state": 42,
        "verbose": 0,
        "early_stopping_rounds": 500, 
        "depth" : trial.suggest_int("depth", 2, 7),
        "l2_leaf_reg" : trial.suggest_int("l2_leaf_reg", 3, 200),
        "class_weights" : [1, weight_1]       
    }

    curr_time = start_time

    f_betas = []

    while curr_time < end_time:
        train_df = data[(data.utc >= curr_time - dt.timedelta(days=35)) & (data.utc <= curr_time - dt.timedelta(days=5))]
        val_df = data[(data.utc > curr_time - dt.timedelta(days=5)) & (data.utc <= curr_time)]
        test_df = data[(data.utc > curr_time) & (data.utc <= curr_time + dt.timedelta(days=5))]
        if test_df.empty:
            break

        X_train, y_train = train_df[cat_feats + num_feats], train_df[target]
        X_val, y_val = val_df[cat_feats + num_feats], val_df[target]
        X_test, y_test = test_df[cat_feats + num_feats], test_df[target]

        model = CatboostFinModel(params)
        model.set_datasets(X_train, X_val, y_train, y_val)
        model.set_features(num_feats, cat_feats)
        model.fit()

        proba = model.predict_proba(X_test)[:, 1]
        preds = (proba > 0.5).astype(int)

        f_betas.append(fbeta_score(y_test, preds, beta=0.0001))
        curr_time += dt.timedelta(days=5)

    final = np.mean(f_betas)

    return final


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=7)

print("Лучшие параметры:")
print(study.best_params)
print("Лучшее значение F-beta:")
print(study.best_value)
 

[I 2025-06-26 12:42:19,209] A new study created in memory with name: no-name-95a8f2c1-2be3-4ac0-a9b5-b7d87cf87cb5
[I 2025-06-26 13:07:49,270] Trial 0 finished with value: 0.6935938920954564 and parameters: {'class_weight_1': 0.504179614103816, 'depth': 6, 'l2_leaf_reg': 75}. Best is trial 0 with value: 0.6935938920954564.
[I 2025-06-26 13:18:54,387] Trial 1 finished with value: 0.7093273691631244 and parameters: {'class_weight_1': 0.4699610912910789, 'depth': 3, 'l2_leaf_reg': 109}. Best is trial 1 with value: 0.7093273691631244.
[I 2025-06-26 13:35:55,179] Trial 2 finished with value: 0.6573680485011472 and parameters: {'class_weight_1': 0.6959749912706961, 'depth': 6, 'l2_leaf_reg': 88}. Best is trial 1 with value: 0.7093273691631244.
[I 2025-06-26 13:44:57,060] Trial 3 finished with value: 0.7144644193505277 and parameters: {'class_weight_1': 0.31244694007955265, 'depth': 2, 'l2_leaf_reg': 17}. Best is trial 3 with value: 0.7144644193505277.
[I 2025-06-26 14:06:57,212] Trial 4 finis

Лучшие параметры:
{'class_weight_1': 0.31244694007955265, 'depth': 2, 'l2_leaf_reg': 17}
Лучшее значение F-beta:
0.7144644193505277


In [13]:
def objective(trial):
    weight_1 = trial.suggest_float("class_weight_1", 0.2, 0.4)
    params={
        "iterations" : 10000,
        "learning_rate" : 0.02,
        "use_best_model" : True,
        "loss_function" : 'Logloss',
        "eval_metric" : 'Logloss',
        "cat_features" : [], 
        "random_state": 42,
        "verbose": 0,
        "early_stopping_rounds": 500, 
        "depth" : trial.suggest_int("depth", 2, 7),
        "l2_leaf_reg" : trial.suggest_int("l2_leaf_reg", 3, 200),
        "class_weights" : [1, weight_1]       
    }

    curr_time = start_time

    f_betas = []

    while curr_time < end_time:
        train_df = data[(data.utc >= curr_time - dt.timedelta(days=35)) & (data.utc <= curr_time - dt.timedelta(days=5))]
        val_df = data[(data.utc > curr_time - dt.timedelta(days=5)) & (data.utc <= curr_time)]
        test_df = data[(data.utc > curr_time) & (data.utc <= curr_time + dt.timedelta(days=5))]
        if test_df.empty:
            break

        X_train, y_train = train_df[cat_feats + num_feats], train_df[target]
        X_val, y_val = val_df[cat_feats + num_feats], val_df[target]
        X_test, y_test = test_df[cat_feats + num_feats], test_df[target]

        model = CatboostFinModel(params)
        model.set_datasets(X_train, X_val, y_train, y_val)
        model.set_features(num_feats, cat_feats)
        model.fit()

        proba = model.predict_proba(X_test)[:, 1]
        preds = (proba > 0.5).astype(int)

        f_betas.append(fbeta_score(y_test, preds, beta=0.0001))
        curr_time += dt.timedelta(days=5)

    final = np.mean(f_betas)

    return final


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=7)

print("Лучшие параметры:")
print(study.best_params)
print("Лучшее значение F-beta:")
print(study.best_value)
 

[I 2025-06-26 17:41:21,255] A new study created in memory with name: no-name-b08f16ba-df29-4ff3-b677-93083b33e953
[I 2025-06-26 17:53:45,329] Trial 0 finished with value: 0.7653131021838674 and parameters: {'class_weight_1': 0.355975286748748, 'depth': 2, 'l2_leaf_reg': 89}. Best is trial 0 with value: 0.7653131021838674.
[I 2025-06-26 18:15:43,099] Trial 1 finished with value: 0.7763703334242507 and parameters: {'class_weight_1': 0.3450620257200212, 'depth': 7, 'l2_leaf_reg': 80}. Best is trial 1 with value: 0.7763703334242507.
[I 2025-06-26 18:28:29,553] Trial 2 finished with value: 0.7270217372309319 and parameters: {'class_weight_1': 0.2291750196882985, 'depth': 5, 'l2_leaf_reg': 132}. Best is trial 1 with value: 0.7763703334242507.
[I 2025-06-26 18:47:34,728] Trial 3 finished with value: 0.705153186857037 and parameters: {'class_weight_1': 0.24370360117842632, 'depth': 7, 'l2_leaf_reg': 69}. Best is trial 1 with value: 0.7763703334242507.
[I 2025-06-26 18:59:01,452] Trial 4 finish

Лучшие параметры:
{'class_weight_1': 0.3450620257200212, 'depth': 7, 'l2_leaf_reg': 80}
Лучшее значение F-beta:
0.7763703334242507
